# CS4.406 A2 — Q3 (baseline/ablation/CI) & Q5 (extended evaluation)

**Input:** `reranked_ebnerd_validation_scores.parquet` and
`reranked_mind_validation_scores.parquet`, handed off from the Q2/Q4 half of
the team. Each row is one impression with the retrieved candidate list,
ground-truth labels, the Stage-1 (retrieval-only) score, and the Stage-2
(re-ranked) score.

**Mapping used for Q3:** "baseline reproduced" = ranking by `stage1_scores`
alone (retrieval score, no click-log-trained signal — the honest analogue of
the starter/official baseline given what data this handoff contains).
"the one principled improvement" = the trained re-ranker's `reranked_scores`
(adds click-log behavioural features on top of retrieval). This is stated
explicitly here so it goes into the design note the same way, rather than
being implied.

**Known gap (flag to your partner, not a blocker for what follows):**
mean candidate-list size is ~12 (EB-NeRD) / ~37 (MIND) — matching your A1
*live-pool* sizes, not a full-catalogue top-150 retrieval. Confirm with him
whether Stage 1 retrieved from the full catalogue or reranked the given pool;
it changes one sentence in the report, nothing in this notebook.

**Known gap (blocks part of Q5 until you get a companion file):**
diversity/novelty/coverage and the cold/warm + head/tail slices need article
category, article popularity, and user history length — none of which are in
this parquet. Section 5 below is written to consume that data the moment you
have it; ask your partner for two small tables:
  - `article_id, category, popularity` (from his article feature store)
  - `user_id, hist_len` (from his `ctx_eval`)


In [32]:
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from pathlib import Path

def find_one(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"could not find {pattern} under /kaggle/input — "
                                 f"check the dataset was actually attached to this notebook")
    return hits[0]

DATASETS = {
    "EB-NeRD": find_one("reranked_ebnerd_validation_scores.parquet"),
    "MIND": find_one("reranked_mind_validation_scores.parquet"),
}


dfs = {name: pl.read_parquet(path) for name, path in DATASETS.items()}
for name, df in dfs.items():
    print(name, df.shape, df.columns)


EB-NeRD (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']
MIND (5000, 6) ['impression_id', 'user_id', 'candidate_ids', 'labels', 'stage1_scores', 'reranked_scores']


In [33]:
import polars as pl
mind_scores = pl.read_parquet(find_one("reranked_mind_validation_scores.parquet"))
sample_ids = mind_scores["candidate_ids"].head(1).to_list()[0]  # one impression's candidate list
print(sample_ids[:5])

['N28682', 'N48740', 'N31958', 'N34130', 'N6916']


## Part I — Q5 core metrics (built first: Q3's ablation is computed from these same per-impression arrays)

In [34]:
# ---------------------------------------------------------------------------
# Per-impression metric functions — standard definitions used by the
# MIND/EB-NeRD official evaluators (binary relevance, AUC/MRR/nDCG).
# ---------------------------------------------------------------------------

def auc_per_impression(labels, scores):
    labels = np.asarray(labels)
    if labels.min() == labels.max():
        return np.nan  # AUC undefined with only one class present
    return roc_auc_score(labels, scores)


def mrr_per_impression(labels, scores):
    order = np.argsort(-np.asarray(scores))
    ranked = np.asarray(labels)[order]
    hits = np.flatnonzero(ranked == 1)
    return np.nan if len(hits) == 0 else 1.0 / (hits[0] + 1)


def _dcg_at_k(labels_sorted, k):
    labels_sorted = np.asarray(labels_sorted[:k], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(labels_sorted) + 2))
    return float(np.sum(labels_sorted * discounts))


def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels)
    order = np.argsort(-np.asarray(scores))
    dcg = _dcg_at_k(labels[order], k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return np.nan if idcg == 0 else dcg / idcg


def per_impression_metrics(df, score_col):
    """Returns a dict of 1D numpy arrays, one value per row/impression,
    aligned to df's row order — this alignment is what makes the PAIRED
    bootstrap in Q3 valid (same impression, same position, both variants).
    """
    aucs, mrrs, ndcg5, ndcg10 = [], [], [], []
    for labels, scores in zip(df["labels"].to_list(), df[score_col].to_list()):
        aucs.append(auc_per_impression(labels, scores))
        mrrs.append(mrr_per_impression(labels, scores))
        ndcg5.append(ndcg_at_k(labels, scores, 5))
        ndcg10.append(ndcg_at_k(labels, scores, 10))
    return dict(auc=np.array(aucs), mrr=np.array(mrrs),
                ndcg5=np.array(ndcg5), ndcg10=np.array(ndcg10))


metrics = {}  # metrics[dataset][stage] -> dict of per-impression arrays
for name, df in dfs.items():
    metrics[name] = {
        "stage1": per_impression_metrics(df, "stage1_scores"),
        "stage2": per_impression_metrics(df, "reranked_scores"),
    }
print("Per-impression metrics computed for both stages, both datasets.")


Per-impression metrics computed for both stages, both datasets.


In [35]:
# ---------------------------------------------------------------------------
# Unpaired bootstrap CI (for the headline "here is our metric with a CI" table)
# ---------------------------------------------------------------------------
def bootstrap_ci(values, n_boot=1000, seed=0):
    values = values[~np.isnan(values)]
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean()
                            for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return float(values.mean()), float(lo), float(hi)


rows = []
for name in DATASETS:
    for stage in ("stage1", "stage2"):
        for metric_name, arr in metrics[name][stage].items():
            mean, lo, hi = bootstrap_ci(arr)
            rows.append({"dataset": name, "stage": stage, "metric": metric_name,
                          "mean": round(mean, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})

summary_table = pl.DataFrame(rows)
print(summary_table)
summary_table.write_csv("q5_metrics_summary.csv")


shape: (16, 6)
┌─────────┬────────┬────────┬────────┬────────┬────────┐
│ dataset ┆ stage  ┆ metric ┆ mean   ┆ ci_lo  ┆ ci_hi  │
│ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str    ┆ f64    ┆ f64    ┆ f64    │
╞═════════╪════════╪════════╪════════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ auc    ┆ 0.4846 ┆ 0.4758 ┆ 0.4939 │
│ EB-NeRD ┆ stage1 ┆ mrr    ┆ 0.3069 ┆ 0.2997 ┆ 0.3146 │
│ EB-NeRD ┆ stage1 ┆ ndcg5  ┆ 0.3331 ┆ 0.3241 ┆ 0.3424 │
│ EB-NeRD ┆ stage1 ┆ ndcg10 ┆ 0.4227 ┆ 0.4156 ┆ 0.4302 │
│ EB-NeRD ┆ stage2 ┆ auc    ┆ 0.6957 ┆ 0.6881 ┆ 0.7036 │
│ …       ┆ …      ┆ …      ┆ …      ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ ndcg10 ┆ 0.342  ┆ 0.3335 ┆ 0.3516 │
│ MIND    ┆ stage2 ┆ auc    ┆ 0.6067 ┆ 0.5991 ┆ 0.6152 │
│ MIND    ┆ stage2 ┆ mrr    ┆ 0.3264 ┆ 0.3176 ┆ 0.3366 │
│ MIND    ┆ stage2 ┆ ndcg5  ┆ 0.3025 ┆ 0.2935 ┆ 0.3132 │
│ MIND    ┆ stage2 ┆ ndcg10 ┆ 0.3669 ┆ 0.3583 ┆ 0.3767 │
└─────────┴────────┴────────┴────────┴────────┴────────┘


## Part I — Q2's required deliverable, produced here since the inputs already exist: before/after re-ranking table

In [36]:
before_after_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        before = np.nanmean(metrics[name]["stage1"][metric_name])
        after = np.nanmean(metrics[name]["stage2"][metric_name])
        before_after_rows.append({
            "dataset": name, "metric": metric_name,
            "before_rerank": round(before, 4), "after_rerank": round(after, 4),
            "delta": round(after - before, 4),
        })

before_after = pl.DataFrame(before_after_rows)
print(before_after)
before_after.write_csv("q2_before_after_rerank.csv")


shape: (8, 5)
┌─────────┬────────┬───────────────┬──────────────┬────────┐
│ dataset ┆ metric ┆ before_rerank ┆ after_rerank ┆ delta  │
│ ---     ┆ ---    ┆ ---           ┆ ---          ┆ ---    │
│ str     ┆ str    ┆ f64           ┆ f64          ┆ f64    │
╞═════════╪════════╪═══════════════╪══════════════╪════════╡
│ EB-NeRD ┆ auc    ┆ 0.4846        ┆ 0.6957       ┆ 0.2111 │
│ EB-NeRD ┆ mrr    ┆ 0.3069        ┆ 0.4578       ┆ 0.1509 │
│ EB-NeRD ┆ ndcg5  ┆ 0.3331        ┆ 0.5172       ┆ 0.1841 │
│ EB-NeRD ┆ ndcg10 ┆ 0.4227        ┆ 0.5683       ┆ 0.1456 │
│ MIND    ┆ auc    ┆ 0.561         ┆ 0.6067       ┆ 0.0457 │
│ MIND    ┆ mrr    ┆ 0.3038        ┆ 0.3264       ┆ 0.0226 │
│ MIND    ┆ ndcg5  ┆ 0.2782        ┆ 0.3025       ┆ 0.0243 │
│ MIND    ┆ ndcg10 ┆ 0.342         ┆ 0.3669       ┆ 0.0249 │
└─────────┴────────┴───────────────┴──────────────┴────────┘


## Part I — Q3. Baseline Reproduced, Then Beaten (paired bootstrap CI)

This is the methodological piece the spec calls out specifically: **"claimed
gains must ship a paired bootstrap 95% CI that excludes zero."** Paired means
the *same* resampled impressions are used for both variants in every
bootstrap draw, so we're testing the CI of the *difference*, not comparing
two separate CIs that happen not to overlap (which is what A1 did — this is
a deliberate upgrade for A2).


In [37]:
def paired_bootstrap_ci_diff(arr_improved, arr_baseline, n_boot=2000, seed=0):
    """Both arrays must be the SAME LENGTH and SAME ROW ORDER (guaranteed
    here since both came from per_impression_metrics on the same dataframe).
    NaNs (undefined AUC on single-class impressions) are dropped pairwise.
    """
    a, b = np.asarray(arr_improved), np.asarray(arr_baseline)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    n = len(a)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)          # SAME resample for both arrays
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(diffs.mean()), float(lo), float(hi), n


ablation_rows = []
for name in DATASETS:
    for metric_name in ("auc", "mrr", "ndcg5", "ndcg10"):
        improved = metrics[name]["stage2"][metric_name]
        baseline = metrics[name]["stage1"][metric_name]
        mean_diff, lo, hi, n_used = paired_bootstrap_ci_diff(improved, baseline)
        significant = lo > 0
        ablation_rows.append({
            "dataset": name, "metric": metric_name, "n_impressions": n_used,
            "mean_delta": round(mean_diff, 4), "ci_lo": round(lo, 4), "ci_hi": round(hi, 4),
            "significant (CI excludes 0)": significant,
        })

ablation_table = pl.DataFrame(ablation_rows)
print(ablation_table)
ablation_table.write_csv("q3_ablation_paired_ci.csv")

print()
for r in ablation_rows:
    verdict = "SIGNIFICANT" if r["significant (CI excludes 0)"] else "not significant — CI includes 0"
    print(f"{r['dataset']:8s} {r['metric']:6s}  Δ={r['mean_delta']:+.4f}  "
          f"CI=[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  -> {verdict}")


shape: (8, 7)
┌─────────┬────────┬───────────────┬────────────┬────────┬────────┬─────────────────────────────┐
│ dataset ┆ metric ┆ n_impressions ┆ mean_delta ┆ ci_lo  ┆ ci_hi  ┆ significant (CI excludes 0) │
│ ---     ┆ ---    ┆ ---           ┆ ---        ┆ ---    ┆ ---    ┆ ---                         │
│ str     ┆ str    ┆ i64           ┆ f64        ┆ f64    ┆ f64    ┆ bool                        │
╞═════════╪════════╪═══════════════╪════════════╪════════╪════════╪═════════════════════════════╡
│ EB-NeRD ┆ auc    ┆ 5000          ┆ 0.2111     ┆ 0.1994 ┆ 0.2232 ┆ true                        │
│ EB-NeRD ┆ mrr    ┆ 5000          ┆ 0.1508     ┆ 0.1393 ┆ 0.162  ┆ true                        │
│ EB-NeRD ┆ ndcg5  ┆ 5000          ┆ 0.1839     ┆ 0.1714 ┆ 0.1963 ┆ true                        │
│ EB-NeRD ┆ ndcg10 ┆ 5000          ┆ 0.1456     ┆ 0.1359 ┆ 0.1551 ┆ true                        │
│ MIND    ┆ auc    ┆ 5000          ┆ 0.0457     ┆ 0.0387 ┆ 0.0524 ┆ true                        │
│ MIND

In [38]:
import polars as pl
from pathlib import Path

def find_one(pattern):
    hits = sorted(Path("/kaggle/input").rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"couldn't find {pattern} under /kaggle/input")
    return hits[0]
def find_by_folder(folder_keyword, filename_pattern):
    hits = sorted(
        p for p in Path("/kaggle/input").rglob(filename_pattern)
        if folder_keyword in p.parent.name
    )
    if not hits:
        raise FileNotFoundError(f"couldn't find {filename_pattern} inside a folder "
                                 f"containing '{folder_keyword}' under /kaggle/input")
    return hits[0]

behaviors_paths = [
    find_by_folder("train", "behaviors.parquet"),
    find_by_folder("validation", "behaviors.parquet"),
]
history_paths = [
    find_by_folder("train", "history.parquet"),
    find_by_folder("validation", "history.parquet"),
]
# ---------------------------------------------------------------------------
# article_meta.parquet — category (direct) + popularity (computed from clicks)
# ---------------------------------------------------------------------------
articles_path = find_one("articles.parquet")   # picks up the ebnerd_large one first match
articles = pl.read_parquet(articles_path)

# EB-NeRD's category column name varies by dump version — handle both
cat_col = "category_str" if "category_str" in articles.columns else "category"

# Click counts from BOTH train and validation behaviour logs, since popularity
# should reflect real click volume, not just the smaller validation split

clicks = (
    pl.concat([pl.scan_parquet(p) for p in behaviors_paths])
    .select("article_ids_clicked")
    .explode("article_ids_clicked")
    .drop_nulls()
    .group_by("article_ids_clicked")
    .agg(pl.len().alias("click_count"))
    .collect()
)

article_meta = (
    articles.select(["article_id", cat_col])
    .rename({cat_col: "category"})
    .join(clicks, left_on="article_id", right_on="article_ids_clicked", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .with_columns(pl.col("article_id").cast(pl.Utf8))   # match string IDs used in the reranked parquet
    .select(["article_id", "category", "popularity"])
)

print(article_meta.head())
article_meta.write_parquet("/kaggle/working/article_meta.parquet")
print("wrote", article_meta.height, "rows to article_meta.parquet")

shape: (5, 3)
┌────────────┬───────────────┬────────────┐
│ article_id ┆ category      ┆ popularity │
│ ---        ┆ ---           ┆ ---        │
│ str        ┆ str           ┆ f64        │
╞════════════╪═══════════════╪════════════╡
│ 3000022    ┆ underholdning ┆ 0.0        │
│ 3000063    ┆ nyheder       ┆ 0.0        │
│ 3000613    ┆ sport         ┆ 0.0        │
│ 3000700    ┆ underholdning ┆ 0.0        │
│ 3000840    ┆ nyheder       ┆ 0.0        │
└────────────┴───────────────┴────────────┘
wrote 125541 rows to article_meta.parquet


In [39]:
# ---------------------------------------------------------------------------
# user_hist_len.parquet — click-history length per user
# ---------------------------------------------------------------------------
hist_all = pl.concat([pl.read_parquet(p) for p in history_paths])

user_hist_len = (
    hist_all.select(["user_id", "article_id_fixed"])
    .with_columns(pl.col("article_id_fixed").list.len().alias("hist_len"))
    .unique(subset="user_id", keep="last")
    .with_columns(pl.col("user_id").cast(pl.Utf8))
    .select(["user_id", "hist_len"])
)

print(user_hist_len.head())
user_hist_len.write_parquet("/kaggle/working/user_hist_len.parquet")
print("wrote", user_hist_len.height, "rows to user_hist_len.parquet")

shape: (5, 2)
┌─────────┬──────────┐
│ user_id ┆ hist_len │
│ ---     ┆ ---      │
│ str     ┆ u32      │
╞═════════╪══════════╡
│ 2304986 ┆ 275      │
│ 1948780 ┆ 204      │
│ 1606080 ┆ 179      │
│ 1741728 ┆ 46       │
│ 2134003 ┆ 5        │
└─────────┴──────────┘
wrote 974791 rows to user_hist_len.parquet


In [40]:
print("articles dtype:", articles["article_id"].dtype)
print("clicks column name and dtype:", clicks.columns, clicks["article_ids_clicked"].dtype)
print("sample click IDs:", clicks["article_ids_clicked"].head(5).to_list())
print("sample article IDs:", article_meta["article_id"].head(5).to_list())

articles dtype: Int32
clicks column name and dtype: ['article_ids_clicked', 'click_count'] Int32
sample click IDs: [9712591, 9691666, 9772169, 9754112, 9773768]
sample article IDs: ['3000022', '3000063', '3000613', '3000700', '3000840']


In [41]:
print(behaviors_paths[0])  # confirm which file this actually came from
print(pl.read_parquet(behaviors_paths[0]).columns)  # list every column in that file

/kaggle/input/datasets/sherleysonali/ebnerd-data/ebnerd_large/train/behaviors.parquet
['impression_id', 'article_id', 'impression_time', 'read_time', 'scroll_percentage', 'device_type', 'article_ids_inview', 'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode', 'age', 'is_subscriber', 'session_id', 'next_read_time', 'next_scroll_percentage']


In [42]:
hist_sample = pl.read_parquet(history_paths[0])["article_id_fixed"].head(3).to_list()
print(hist_sample)

[[9735579, 9739888, 9739471, 9739864, 9738441, 9739883, 9739153, 9739844, 9739634, 9739802, 9647575, 9739597, 9739640, 9736662, 9740427, 9740223, 9740013, 9740432, 9740356, 9740484, 9740174, 9241111, 9740182, 9740323, 9740405, 9740249, 9740106, 9740236, 9738800, 9739938, 9738949, 9740969, 9740957, 9740962, 9740749, 9740853, 9740843, 9740905, 9738585, 9741449, 9741476, 9741428, 9741360, 9741106, 9741311, 9740903, 9738801, 9741772, 9741728, 9741218, 9741578, 9741589, 9741415, 9739272, 9742675, 9740408, 9742675, 9667501, 9742675, 9742627, 9742675, 9742675, 9742586, 9742675, 9742225, 9742675, 9742625, 9742675, 9741997, 9742675, 9742634, 9742675, 9741144, 9742675, 9742619, 9742675, 9742172, 9742675, 9740324, 9742675, 9742509, 9742675, 9742270, 9742675, 9742675, 9742560, 9742675, 9742635, 9742748, 9742693, 9733616, 9742681, 9740576, 9743090, 9736862, 9735166, 9742173, 9742885, 9743002, 9743099, 9742735, 9743574, 9742401, 9743572, 9743195, 9743320, 9742073, 9743490, 9743523, 9742764, 9743748,

In [43]:
clicks = (
    pl.concat([pl.scan_parquet(p) for p in behaviors_paths])   # behaviors, NOT history
    .select("article_ids_clicked")
    .explode("article_ids_clicked")
    .drop_nulls()
    .group_by("article_ids_clicked")
    .agg(pl.len().alias("click_count"))
    .collect()
)

print(clicks.head())
print("sample click IDs:", clicks["article_ids_clicked"].head(5).to_list())

shape: (5, 2)
┌─────────────────────┬─────────────┐
│ article_ids_clicked ┆ click_count │
│ ---                 ┆ ---         │
│ i32                 ┆ u32         │
╞═════════════════════╪═════════════╡
│ 9777007             ┆ 2654        │
│ 9649859             ┆ 1           │
│ 9774557             ┆ 5903        │
│ 9600210             ┆ 1           │
│ 8638439             ┆ 3           │
└─────────────────────┴─────────────┘
sample click IDs: [9777007, 9649859, 9774557, 9600210, 8638439]


In [44]:
article_meta = (
    articles.select(["article_id", cat_col])
    .rename({cat_col: "category"})
    .with_columns(pl.col("article_id").cast(pl.Utf8))
    .join(clicks.with_columns(pl.col("article_ids_clicked").cast(pl.Utf8)),
          left_on="article_id", right_on="article_ids_clicked", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)
print(article_meta.filter(pl.col("popularity") > 0).height, "articles with nonzero popularity")

6599 articles with nonzero popularity


In [45]:
# flatten every candidate_id that appears in the actual eval data
all_candidates = (
    dfs["EB-NeRD"]["candidate_ids"]
    .explode()
    .unique()
)
pop_lookup = dict(zip(article_meta["article_id"], article_meta["popularity"]))

n_total = all_candidates.len()
n_covered = sum(1 for a in all_candidates.to_list() if pop_lookup.get(a, 0.0) > 0)
print(f"{n_covered} / {n_total} candidate articles have nonzero popularity ({100*n_covered/n_total:.1f}%)")

1916 / 1934 candidate articles have nonzero popularity (99.1%)


In [46]:
news = pl.read_csv(find_one("news.tsv"), separator="\t", has_header=False,
                    columns=[0, 1], new_columns=["article_id", "category"])
behaviors_path = find_by_folder("dev", "behaviors.tsv")
behaviors = pl.read_csv(behaviors_path, separator="\t", has_header=False,
                         new_columns=["imp_id", "user_id", "time", "history", "impressions"])

In [47]:
# ---------------------------------------------------------------------------
# user_hist_len.parquet — click-history length per user
# ---------------------------------------------------------------------------
user_hist_len = (
    behaviors.select(["user_id", "history"])
    .unique(subset="user_id", keep="last")
    .with_columns(
        pl.col("history").fill_null("").str.split(" ").list.len().alias("hist_len")
    )
    .select(["user_id", "hist_len"])
)

print(user_hist_len.head())
user_hist_len.write_parquet("/kaggle/working/user_hist_len_mind.parquet")
print("wrote", user_hist_len.height, "rows")

shape: (5, 2)
┌─────────┬──────────┐
│ user_id ┆ hist_len │
│ ---     ┆ ---      │
│ str     ┆ u32      │
╞═════════╪══════════╡
│ U265811 ┆ 14       │
│ U184184 ┆ 81       │
│ U467130 ┆ 38       │
│ U629818 ┆ 14       │
│ U742822 ┆ 6        │
└─────────┴──────────┘
wrote 255990 rows


In [48]:
print(article_meta["article_id"].dtype)
print(article_meta["article_id"].head(5).to_list())

String
['3000022', '3000063', '3000613', '3000700', '3000840']


In [49]:
print(article_meta["article_id"].dtype)
print(article_meta["article_id"].head(5).to_list())

String
['3000022', '3000063', '3000613', '3000700', '3000840']


In [50]:
clicked = (
    behaviors.select("impressions")
    .with_columns(pl.col("impressions").str.split(" "))
    .explode("impressions")
    .filter(pl.col("impressions").str.ends_with("-1"))
    .with_columns(pl.col("impressions").str.replace("-1", "").alias("article_id"))
    .group_by("article_id")
    .agg(pl.len().alias("click_count"))
)

article_meta_mind = (
    news.join(clicked, on="article_id", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)

print(article_meta_mind.head())
print(article_meta_mind.filter(pl.col("popularity") > 0).height, "/ ", article_meta_mind.height, "articles with nonzero popularity")

article_meta_mind.write_parquet("article_meta_mind.parquet")
print("wrote article_meta_mind.parquet")

shape: (5, 3)
┌────────────┬───────────┬────────────┐
│ article_id ┆ category  ┆ popularity │
│ ---        ┆ ---       ┆ ---        │
│ str        ┆ str       ┆ f64        │
╞════════════╪═══════════╪════════════╡
│ N55528     ┆ lifestyle ┆ 0.0        │
│ N19639     ┆ health    ┆ 0.0        │
│ N61837     ┆ news      ┆ 0.0        │
│ N53526     ┆ health    ┆ 0.0        │
│ N38324     ┆ health    ┆ 0.0        │
└────────────┴───────────┴────────────┘
738 /  25238 articles with nonzero popularity
wrote article_meta_mind.parquet


In [51]:
all_candidates_mind = dfs["MIND"]["candidate_ids"].explode().unique()
pop_lookup_mind = dict(zip(article_meta_mind["article_id"], article_meta_mind["popularity"]))

n_total = all_candidates_mind.len()
n_covered = sum(1 for a in all_candidates_mind.to_list() if pop_lookup_mind.get(a, 0.0) > 0)
print(f"{n_covered} / {n_total} candidate articles have nonzero popularity ({100*n_covered/n_total:.1f}%)")

22 / 2918 candidate articles have nonzero popularity (0.8%)


In [52]:
news_ids = set(news["article_id"].to_list())
all_candidates_list = all_candidates_mind.to_list()
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in news.tsv at all")

777 / 2918 candidate articles exist in news.tsv at all


In [53]:
print(find_one("news.tsv"))

/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train/news.tsv


In [54]:
news_path = find_by_folder("dev", "news.tsv")
print(news_path)   # confirm this now points into your mind-large-dev upload

news = pl.read_csv(news_path, separator="\t", has_header=False,
                    columns=[0, 1], new_columns=["article_id", "category"])

/kaggle/input/datasets/sherleysonali/mind-large-dev/MINDlarge_dev/news.tsv


In [55]:
clicked = (
    behaviors.select("impressions")
    .with_columns(pl.col("impressions").str.split(" "))
    .explode("impressions")
    .filter(pl.col("impressions").str.ends_with("-1"))
    .with_columns(pl.col("impressions").str.replace("-1", "").alias("article_id"))
    .group_by("article_id")
    .agg(pl.len().alias("click_count"))
)

article_meta_mind = (
    news.join(clicked, on="article_id", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)

print(article_meta_mind.head())
print(article_meta_mind.filter(pl.col("popularity") > 0).height, "/ ", article_meta_mind.height, "articles with nonzero popularity")

article_meta_mind.write_parquet("article_meta_mind.parquet")
print("wrote article_meta_mind.parquet")

shape: (5, 3)
┌────────────┬───────────┬────────────┐
│ article_id ┆ category  ┆ popularity │
│ ---        ┆ ---       ┆ ---        │
│ str        ┆ str       ┆ f64        │
╞════════════╪═══════════╪════════════╡
│ N88753     ┆ lifestyle ┆ 0.0        │
│ N23144     ┆ health    ┆ 0.0        │
│ N86255     ┆ health    ┆ 0.0        │
│ N93187     ┆ news      ┆ 0.0        │
│ N75236     ┆ health    ┆ 0.0        │
└────────────┴───────────┴────────────┘
1852 /  35084 articles with nonzero popularity
wrote article_meta_mind.parquet


In [56]:
news_ids = set(news["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in news.tsv at all")

801 / 2918 candidate articles exist in news.tsv at all


In [57]:
print(news_path)

/kaggle/input/datasets/sherleysonali/mind-large-dev/MINDlarge_dev/news.tsv


In [58]:
# ---------------------------------------------------------------------------
# Load MINDlarge_train's news.tsv and union with the dev catalogue you already have
# ---------------------------------------------------------------------------
news_train_path = find_by_folder("train", "news.tsv")
print(news_train_path)   # confirm it resolves into mind-large-train, not the small dataset again

news_train = pl.read_csv(news_train_path, separator="\t", has_header=False,
                          columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news, news_train]).unique(subset="article_id", keep="first")
print(news_full.height, "total distinct articles across train+dev")

news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in the combined catalogue "
      f"({100*n_in_news/len(all_candidates_list):.1f}%)")

/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train/news.tsv
53515 total distinct articles across train+dev
1379 / 2918 candidate articles exist in the combined catalogue (47.3%)


In [59]:
def find_by_dataset(dataset_keyword, filename_pattern):
    hits = sorted(
        p for p in Path("/kaggle/input").rglob(filename_pattern)
        if dataset_keyword in str(p)
    )
    if not hits:
        raise FileNotFoundError(f"couldn't find {filename_pattern} under a path "
                                 f"containing '{dataset_keyword}'")
    return hits[0]

news_train_path = find_by_dataset("mind-large-train", "news.tsv")
print(news_train_path)   # must now show .../mind-large-train/... in the path

/kaggle/input/datasets/sherleysonali/mind-large-train/MINDlarge_train/news.tsv


In [60]:
news_small_dev_path = find_by_dataset("mind-small-dev", "news.tsv")
print(news_small_dev_path)   # confirm it points into mind-small-dev, not another dataset

news_small_dev = pl.read_csv(news_small_dev_path, separator="\t", has_header=False,
                              columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news, news_train, news_small_dev]).unique(subset="article_id", keep="first")
print(news_full.height, "total distinct articles across train+large_dev+small_dev")

news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in the combined catalogue "
      f"({100*n_in_news/len(all_candidates_list):.1f}%)")

/kaggle/input/datasets/sherleysonali/mind-small-dev/MINDsmall_dev/news.tsv
63203 total distinct articles across train+large_dev+small_dev
2148 / 2918 candidate articles exist in the combined catalogue (73.6%)


In [61]:
# ============================================================================
# ONE clean, consolidated MIND article_meta build — replaces all the earlier
# scattered 'news'/'clicked' cells. Distinct names throughout so nothing
# overwrites anything else.
# ============================================================================

# --- catalogues (3 sources, kept distinct) ---
news_dev = pl.read_csv(find_by_folder("dev", "news.tsv"), separator="\t", has_header=False,
                        columns=[0, 1], new_columns=["article_id", "category"])

news_train = pl.read_csv(find_by_dataset("mind-large-train", "news.tsv"), separator="\t",
                          has_header=False, columns=[0, 1], new_columns=["article_id", "category"])

news_small_dev = pl.read_csv(find_by_dataset("mind-small-dev", "news.tsv"), separator="\t",
                              has_header=False, columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news_dev, news_train, news_small_dev]).unique(subset="article_id", keep="first")
print(news_full.height, "total distinct articles across dev+train+small_dev")

# --- coverage check #1: does the catalogue even contain the candidate IDs? ---
all_candidates_mind = dfs["MIND"]["candidate_ids"].explode().unique()
all_candidates_list = all_candidates_mind.to_list()
news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist in the combined catalogue "
      f"({100*n_in_news/len(all_candidates_list):.1f}%)")

# --- click counts (3 sources, kept distinct) ---
def extract_clicks(beh_df):
    return (
        beh_df.select("impressions")
        .with_columns(pl.col("impressions").str.split(" "))
        .explode("impressions")
        .filter(pl.col("impressions").str.ends_with("-1"))
        .with_columns(pl.col("impressions").str.replace("-1", "").alias("article_id"))
        .group_by("article_id")
        .agg(pl.len().alias("click_count"))
    )

behaviors_dev = pl.read_csv(find_by_folder("dev", "behaviors.tsv"), separator="\t", has_header=False,
                             new_columns=["imp_id", "user_id", "time", "history", "impressions"])
behaviors_train = pl.read_csv(find_by_dataset("mind-large-train", "behaviors.tsv"), separator="\t",
                               has_header=False,
                               new_columns=["imp_id", "user_id", "time", "history", "impressions"])
behaviors_small_dev = pl.read_csv(find_by_dataset("mind-small-dev", "behaviors.tsv"), separator="\t",
                                   has_header=False,
                                   new_columns=["imp_id", "user_id", "time", "history", "impressions"])

clicked_dev = extract_clicks(behaviors_dev)
clicked_train = extract_clicks(behaviors_train)
clicked_small_dev = extract_clicks(behaviors_small_dev)

clicked_combined = (
    pl.concat([clicked_dev, clicked_train, clicked_small_dev])
    .group_by("article_id")
    .agg(pl.col("click_count").sum())
)

# --- final article_meta_mind ---
article_meta_mind = (
    news_full.join(clicked_combined, on="article_id", how="left")
    .with_columns(pl.col("click_count").fill_null(0))
    .with_columns((pl.col("click_count") / pl.col("click_count").sum()).alias("popularity"))
    .select(["article_id", "category", "popularity"])
)

pop_lookup_mind = dict(zip(article_meta_mind["article_id"], article_meta_mind["popularity"]))
n_covered = sum(1 for a in all_candidates_list if pop_lookup_mind.get(a, 0.0) > 0)
print(f"{n_covered} / {len(all_candidates_list)} candidate articles have nonzero popularity "
      f"({100*n_covered/len(all_candidates_list):.1f}%)")

article_meta_mind.write_parquet("article_meta_mind.parquet")
print("wrote article_meta_mind.parquet")

78900 total distinct articles across dev+train+small_dev
2256 / 2918 candidate articles exist in the combined catalogue (77.3%)
1553 / 2918 candidate articles have nonzero popularity (53.2%)
wrote article_meta_mind.parquet


In [62]:
news_test_path = find_by_dataset("mind-large-test-news-tsv", "news.tsv")  # match your actual dataset name
print(news_test_path)  # confirm it resolves correctly before trusting it

news_test = pl.read_csv(news_test_path, separator="\t", has_header=False,
                         columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news_dev, news_train, news_small_dev, news_test]).unique(
    subset="article_id", keep="first")
print(news_full.height, "total distinct articles across dev+train+small_dev+test")

news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist ({100*n_in_news/len(all_candidates_list):.1f}%)")

/kaggle/input/datasets/sherleysonali/mind-large-test-news-tsv/news.tsv
102840 total distinct articles across dev+train+small_dev+test
2562 / 2918 candidate articles exist (87.8%)


In [63]:
your_actual_dataset_name = "mind-large-test-news-tsv"   # <-- replace with EXACTLY what the panel shows

news_test_path = find_by_dataset(your_actual_dataset_name, "news.tsv")
print(news_test_path)   # MUST show your own upload path — check before continuing

/kaggle/input/datasets/sherleysonali/mind-large-test-news-tsv/news.tsv


In [64]:
news_test = pl.read_csv(news_test_path, separator="\t", has_header=False,
                         columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news_dev, news_train, news_small_dev, news_test]).unique(
    subset="article_id", keep="first")
print(news_full.height, "total distinct articles")

news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist ({100*n_in_news/len(all_candidates_list):.1f}%)")

102840 total distinct articles
2562 / 2918 candidate articles exist (87.8%)


In [65]:
news_small_train_path = find_by_dataset("arashnic", "news.tsv")  # confirm this is the small_train one
print(news_small_train_path)

news_small_train = pl.read_csv(news_small_train_path, separator="\t", has_header=False,
                                columns=[0, 1], new_columns=["article_id", "category"])

news_full = pl.concat([news_dev, news_train, news_small_dev, news_test, news_small_train]).unique(
    subset="article_id", keep="first")
print(news_full.height, "total distinct articles — now covering ALL four of his config.py splits")

news_ids = set(news_full["article_id"].to_list())
n_in_news = sum(1 for a in all_candidates_list if a in news_ids)
print(f"{n_in_news} / {len(all_candidates_list)} candidate articles exist ({100*n_in_news/len(all_candidates_list):.1f}%)")

/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train/news.tsv
107447 total distinct articles — now covering ALL four of his config.py splits
2666 / 2918 candidate articles exist (91.4%)


## Part I — Q5. Extended Evaluation — coverage/diversity/novelty and slices

**Blocked on a companion file from your partner** (see the note at the top).
Once you have `article_meta` (columns: `article_id`, `category`,
`popularity`) and `user_hist_len` (columns: `user_id`, `hist_len`), run the
cell below unmodified — it's written against exactly that shape.


In [66]:
# ---------------------------------------------------------------------------
# Fill these in once your partner sends the companion tables. Until then this
# cell will raise a clear FileNotFoundError rather than silently skip.
# ---------------------------------------------------------------------------
try:
    article_meta = pl.read_parquet("article_meta.parquet")      # article_id, category, popularity
    user_hist_len = pl.read_parquet("user_hist_len.parquet")    # user_id, hist_len
    HAVE_METADATA = True
except FileNotFoundError:
    print("article_meta.parquet / user_hist_len.parquet not found yet — "
          "ask your partner for these two small tables, then re-run this cell.")
    HAVE_METADATA = False


In [67]:
# ---------------------------------------------------------------------------
# Coverage / novelty (need popularity); diversity (needs category, as a
# cheap proxy for topical similarity — swap in embedding cosine if your
# partner can also export the 64-d LSA/EMB vectors per article_id).
# ---------------------------------------------------------------------------
def top5_beyond_accuracy(df, score_col, article_meta_df, n_catalogue):
    pop_map = dict(zip(article_meta_df["article_id"], article_meta_df["popularity"]))
    cat_map = dict(zip(article_meta_df["article_id"], article_meta_df["category"]))

    seen_articles = set()
    novelty_scores, diversity_scores = [], []

    for cand_ids, scores in zip(df["candidate_ids"].to_list(), df[score_col].to_list()):
        k = min(5, len(cand_ids))                     # <-- fix: don't assume 5 exist
        order = np.argsort(-np.asarray(scores))[:k]
        top_k = [cand_ids[i] for i in order]
        seen_articles.update(top_k)

        pops = [pop_map.get(a, 0.0) for a in top_k]
        novelty_scores.append(np.mean([-np.log2(p + 1e-9) for p in pops]) if pops else np.nan)

        cats = [cat_map.get(a) for a in top_k]
        pairs = [(cats[i] != cats[j]) for i in range(len(cats)) for j in range(i + 1, len(cats))]
        diversity_scores.append(np.mean(pairs) if pairs else np.nan)

    coverage = len(seen_articles) / n_catalogue
    return dict(novelty=float(np.nanmean(novelty_scores)),
                diversity=float(np.nanmean(diversity_scores)),
                coverage=coverage)


# --- rebuild the loop using PER-DATASET metadata, not one shared `article_meta` ---
article_meta_by_dataset = {
    "EB-NeRD": article_meta,        # your original EB-NeRD table
    "MIND": article_meta_mind,      # the one you just finished fixing
}
n_catalogue_by_dataset = {
    "EB-NeRD": article_meta["article_id"].n_unique(),
    "MIND": article_meta_mind["article_id"].n_unique(),
}

beyond_rows = []
for name in DATASETS:
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        r = top5_beyond_accuracy(dfs[name], col,
                                  article_meta_by_dataset[name],
                                  n_catalogue_by_dataset[name])
        r.update(dataset=name, stage=stage)
        beyond_rows.append(r)

beyond_table = pl.DataFrame(beyond_rows)
print(beyond_table)
beyond_table.write_csv("q5_beyond_accuracy..csv")

shape: (4, 5)
┌───────────┬───────────┬──────────┬─────────┬────────┐
│ novelty   ┆ diversity ┆ coverage ┆ dataset ┆ stage  │
│ ---       ┆ ---       ┆ ---      ┆ ---     ┆ ---    │
│ f64       ┆ f64       ┆ f64      ┆ str     ┆ str    │
╞═══════════╪═══════════╪══════════╪═════════╪════════╡
│ 11.394493 ┆ 0.77534   ┆ 0.01253  ┆ EB-NeRD ┆ stage1 │
│ 10.75017  ┆ 0.72322   ┆ 0.010961 ┆ EB-NeRD ┆ stage2 │
│ 16.584895 ┆ 0.86454   ┆ 0.015856 ┆ MIND    ┆ stage1 │
│ 16.253447 ┆ 0.85006   ┆ 0.015272 ┆ MIND    ┆ stage2 │
└───────────┴───────────┴──────────┴─────────┴────────┘


In [68]:
# ---------------------------------------------------------------------------
# Slices: cold-start vs warm (needs hist_len), head vs tail (needs popularity
# of the CLICKED article specifically).
# ---------------------------------------------------------------------------
def sliced_auc(df, score_col, hist_map, pop_map, cold_thresh=5, head_quantile=0.8):
    aucs_cold, aucs_warm, aucs_head, aucs_tail = [], [], [], []

    pops_all = np.array(list(pop_map.values()))
    head_thr = np.quantile(pops_all, head_quantile) if len(pops_all) else 0.0

    for row in df.iter_rows(named=True):
        labels, scores = row["labels"], row[score_col]
        auc = auc_per_impression(labels, scores)
        hlen = hist_map.get(row["user_id"], 0)
        (aucs_cold if hlen < cold_thresh else aucs_warm).append(auc)

        clicked_idx = [i for i, l in enumerate(labels) if l == 1]
        if clicked_idx:
            clicked_pop = np.mean([pop_map.get(row["candidate_ids"][i], 0.0) for i in clicked_idx])
            (aucs_head if clicked_pop >= head_thr else aucs_tail).append(auc)

    def _summ(a):
        a = np.array([x for x in a if not np.isnan(x)])
        return (np.nan, np.nan) if len(a) == 0 else (a.mean(), len(a))

    return dict(cold=_summ(aucs_cold), warm=_summ(aucs_warm),
                head=_summ(aucs_head), tail=_summ(aucs_tail))


if HAVE_METADATA:
    hist_map = dict(zip(user_hist_len["user_id"], user_hist_len["hist_len"]))
    pop_map_full = dict(zip(article_meta["article_id"], article_meta["popularity"]))

    slice_rows = []
    for name in DATASETS:
        for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
            s = sliced_auc(dfs[name], col, hist_map, pop_map_full)
            for slice_name, (auc, n) in s.items():
                slice_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                     "auc": round(auc, 4) if not np.isnan(auc) else None, "n": n})
    slice_table = pl.DataFrame(slice_rows)
    print(slice_table)
    slice_table.write_csv("q5_slices.csv")


shape: (16, 5)
┌─────────┬────────┬───────┬────────┬────────┐
│ dataset ┆ stage  ┆ slice ┆ auc    ┆ n      │
│ ---     ┆ ---    ┆ ---   ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str   ┆ f64    ┆ f64    │
╞═════════╪════════╪═══════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ cold  ┆ null   ┆ NaN    │
│ EB-NeRD ┆ stage1 ┆ warm  ┆ 0.4846 ┆ 5000.0 │
│ EB-NeRD ┆ stage1 ┆ head  ┆ 0.4846 ┆ 5000.0 │
│ EB-NeRD ┆ stage1 ┆ tail  ┆ null   ┆ NaN    │
│ EB-NeRD ┆ stage2 ┆ cold  ┆ null   ┆ NaN    │
│ …       ┆ …      ┆ …     ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ tail  ┆ null   ┆ NaN    │
│ MIND    ┆ stage2 ┆ cold  ┆ 0.6067 ┆ 5000.0 │
│ MIND    ┆ stage2 ┆ warm  ┆ null   ┆ NaN    │
│ MIND    ┆ stage2 ┆ head  ┆ 0.6067 ┆ 5000.0 │
│ MIND    ┆ stage2 ┆ tail  ┆ null   ┆ NaN    │
└─────────┴────────┴───────┴────────┴────────┘


In [73]:
user_hist_len_mind = pl.read_parquet("user_hist_len_mind.parquet")

mind_users_in_eval = set(dfs["MIND"]["user_id"].to_list())
mind_users_in_hist = set(user_hist_len_mind["user_id"].to_list())
overlap = len(mind_users_in_eval & mind_users_in_hist)
print(f"{overlap} / {len(mind_users_in_eval)} eval users found in user_hist_len_mind "
      f"({100*overlap/len(mind_users_in_eval):.1f}%)")

1431 / 4834 eval users found in user_hist_len_mind (29.6%)


In [72]:
import os
print([f for f in os.listdir(".") if "hist" in f.lower()])

['user_hist_len_mind.parquet', 'user_hist_len.parquet']


In [74]:
def hist_len_from_behaviors(beh_df):
    return (
        beh_df.select(["user_id", "history"])
        .unique(subset="user_id", keep="last")
        .with_columns(pl.col("history").fill_null("").str.split(" ").list.len().alias("hist_len"))
        .select(["user_id", "hist_len"])
    )

hl_dev = hist_len_from_behaviors(behaviors_dev)
hl_train = hist_len_from_behaviors(behaviors_train)
hl_small_dev = hist_len_from_behaviors(behaviors_small_dev)

user_hist_len_mind = (
    pl.concat([hl_dev, hl_train, hl_small_dev])
    .unique(subset="user_id", keep="last")
)
print(user_hist_len_mind.height, "users")

mind_users_in_hist = set(user_hist_len_mind["user_id"].to_list())
overlap = len(mind_users_in_eval & mind_users_in_hist)
print(f"{overlap} / {len(mind_users_in_eval)} eval users found now "
      f"({100*overlap/len(mind_users_in_eval):.1f}%)")

user_hist_len_mind.write_parquet("user_hist_len_mind.parquet")

750434 users
4834 / 4834 eval users found now (100.0%)


In [75]:
def sliced_auc(df, score_col, hist_map, pop_map, cold_thresh=5, head_quantile=0.8):
    aucs_cold, aucs_warm, aucs_head, aucs_tail = [], [], [], []

    live_pops = np.array([p for p in pop_map.values() if p > 0])
    head_thr = np.quantile(live_pops, head_quantile) if len(live_pops) else 0.0

    for row in df.iter_rows(named=True):
        labels, scores = row["labels"], row[score_col]
        auc = auc_per_impression(labels, scores)
        hlen = hist_map.get(row["user_id"], 0)
        (aucs_cold if hlen < cold_thresh else aucs_warm).append(auc)

        clicked_idx = [i for i, l in enumerate(labels) if l == 1]
        if clicked_idx:
            clicked_pop = np.mean([pop_map.get(row["candidate_ids"][i], 0.0) for i in clicked_idx])
            (aucs_head if clicked_pop >= head_thr else aucs_tail).append(auc)

    def _summ(a):
        a = np.array([x for x in a if not np.isnan(x)])
        return (np.nan, np.nan) if len(a) == 0 else (a.mean(), len(a))

    return dict(cold=_summ(aucs_cold), warm=_summ(aucs_warm),
                head=_summ(aucs_head), tail=_summ(aucs_tail))

In [76]:
hist_map_by_dataset = {
    "EB-NeRD": dict(zip(user_hist_len["user_id"], user_hist_len["hist_len"])),
    "MIND": dict(zip(user_hist_len_mind["user_id"], user_hist_len_mind["hist_len"])),
}
pop_map_by_dataset = {
    "EB-NeRD": dict(zip(article_meta["article_id"], article_meta["popularity"])),
    "MIND": dict(zip(article_meta_mind["article_id"], article_meta_mind["popularity"])),
}

slice_rows = []
for name in DATASETS:
    for stage, col in [("stage1", "stage1_scores"), ("stage2", "reranked_scores")]:
        s = sliced_auc(dfs[name], col, hist_map_by_dataset[name], pop_map_by_dataset[name])
        for slice_name, (auc, n) in s.items():
            slice_rows.append({"dataset": name, "stage": stage, "slice": slice_name,
                                 "auc": round(auc, 4) if not np.isnan(auc) else None, "n": n})

slice_table = pl.DataFrame(slice_rows)
print(slice_table)
slice_table.write_csv("q5_slices.csv")

shape: (16, 5)
┌─────────┬────────┬───────┬────────┬────────┐
│ dataset ┆ stage  ┆ slice ┆ auc    ┆ n      │
│ ---     ┆ ---    ┆ ---   ┆ ---    ┆ ---    │
│ str     ┆ str    ┆ str   ┆ f64    ┆ f64    │
╞═════════╪════════╪═══════╪════════╪════════╡
│ EB-NeRD ┆ stage1 ┆ cold  ┆ null   ┆ NaN    │
│ EB-NeRD ┆ stage1 ┆ warm  ┆ 0.4846 ┆ 5000.0 │
│ EB-NeRD ┆ stage1 ┆ head  ┆ 0.4828 ┆ 4621.0 │
│ EB-NeRD ┆ stage1 ┆ tail  ┆ 0.5059 ┆ 379.0  │
│ EB-NeRD ┆ stage2 ┆ cold  ┆ null   ┆ NaN    │
│ …       ┆ …      ┆ …     ┆ …      ┆ …      │
│ MIND    ┆ stage1 ┆ tail  ┆ 0.6118 ┆ 1044.0 │
│ MIND    ┆ stage2 ┆ cold  ┆ 0.5604 ┆ 699.0  │
│ MIND    ┆ stage2 ┆ warm  ┆ 0.6142 ┆ 4301.0 │
│ MIND    ┆ stage2 ┆ head  ┆ 0.6002 ┆ 3956.0 │
│ MIND    ┆ stage2 ┆ tail  ┆ 0.6313 ┆ 1044.0 │
└─────────┴────────┴───────┴────────┴────────┘


In [77]:
cold_thresh = 5
n_cold_ebnerd = sum(1 for h in user_hist_len["hist_len"].to_list() if h < cold_thresh)
print(f"{n_cold_ebnerd} EB-NeRD users with hist_len < {cold_thresh}")

0 EB-NeRD users with hist_len < 5


## Summary of what's committed by this notebook

| File written | Answers |
|---|---|
| `q5_metrics_summary.csv` | Q5 — AUC/MRR/nDCG@5/10, both stages, both datasets, with CIs |
| `q2_before_after_rerank.csv` | Q2.4 — "report metrics before and after re-ranking" |
| `q3_ablation_paired_ci.csv` | Q3 — the paired-bootstrap significance test the spec requires |
| `q5_beyond_accuracy.csv` | Q5 — diversity/novelty/coverage (once companion file arrives) |
| `q5_slices.csv` | Q5 — cold/warm + head/tail slices (once companion file arrives) |

**Still outside this notebook's scope, and still your partner's side:**
Codabench submission of the actual test-set predictions (this parquet is the
*validation* set only) — that inference + submission pipeline stays with
whoever built the Stage-1/Stage-2 scoring functions, since it needs the live
test-set candidate generation, not just these saved validation scores.
